# LABOR-7: FPC1500 — RF-Mischer (Spektrum-Analyzer)

### Ziel

Untersuchung eines **RF-Mischers** mit dem **R&S FPC1500** im Modus **Spectrum Analyzer**.
Es werden **nacheinander drei Spektren** aufgezeichnet, um den **LO** (Lokaloszillator), das **IF-Signal**
und den **RF-Ausgang** des Mischers (bzw. das dort sichtbare Spektrum einschließlich Mischprodukten)
zu beurteilen.

### Wichtig: Messreihenfolge

**Es sind drei Messungen in unmittelbarer Abfolge auszuführen** (zuerst LO, dann IF, dann RF-Ausgang).
Zwischen den Messungen wird die **Messkonfiguration am Aufbau** geändert (Kabel/Führung zum Spektrumanalysator,
ggf. Span/Referenzpegel am Gerät). Das Notebook speichert nach jeder Live-Messung die Daten in eine gemeinsame Replay-Datei.

### Kurz zum Mischer

Der Mischer multipliziert Signale nichtlinear; im Spektrum erscheinen **LO**, **IF** und **RF** sowie **Mischprodukte**
(bei realen Messungen oft zusätzliche Pegel durch Reflexionen und Kalibrierung). Für die Auswertung werden die
**Pegel in dBm** aus `TRACE1` (Leistungswerte) verwendet.

### Darstellung (Plots)

Die drei Spektren nutzen **eigene Figure-IDs** (`mixer_lo` / `mixer_if` / `mixer_rf`), damit sie sich nicht mit anderen matplotlib-Figuren überschneiden. Beim Speichern erscheint eine **Verifikationszeile** mit allen drei Spektren in der JSON-Datei. Marker-Steuerung: **`ipywidgets.IntSlider`** (Trace-Bin) unterhalb des Plots, plus **`-1`/`+1` Buttons** für Schrittbewegungen um genau einen Bin (nützlich zur exakten Peaksuche). Nicht `matplotlib.widgets.Slider`, der mit **ipympl** in Jupyter/VS Code oft zu eingefrorenen Controls führt. Voraussetzungen: **`pip install ipympl`** (`%matplotlib widget`) und **`pip install ipywidgets`**. **`plt.ioff()`** vor dem Erzeugen der Figure verhindert die automatische ipympl-Zweitausgabe; zusammen mit **`display(VBox([fig.canvas, slider, buttons]))`** sonst **doppelter Plot**. Für nur statische Bilder testweise **`%matplotlib inline`**.


## Gerät: Spektrum-Analyzer (Kurzinfo)

- **Mode:** *Spectrum Analyzer* am FPC1500 wählen.
- **Freq / Span:** Span so setzen, dass der erwartete Peak (LO, IF bzw. RF/Mischprodukte) gut sichtbar ist.
- **BW:** RBW/VBW nach Bedarf (feinere RBW → langsamer, schärfere Peaks).
- **Ampt:** Reference Level so wählen, dass der Trace nicht clippt.

Netzwerk: IP **`192.168.1.10`**, SCPI-Port **`5555`** (wie in den anderen Labor-Notebooks).


## Frequency-Mixer ZX05-30W-S+ 

[Mini-Circuits Produktseite (ZX05-30W-S+)](https://www.minicircuits.com/WebStore/dashboard.html?model=ZX05-30W-S%2B)

[Direktlink PDF-Datenblatt](https://www.minicircuits.com/pdfs/ZX05-30W-S%2B.pdf)

## RF-Oszillator Joselin  Breitband-VCO 515 MHz-1150 MHz

[Amazon Produktseite](https://www.amazon.de/dp/B0DZCP56KJ?ref=ppx_yo2ov_dt_b_fed_asin_title)

![](media/rf-mixer.png)

## IF-Signal Generator:  10K-22MHz VFO/RF Generator (SDR QRP HAM Transmitter)

![](media/IF-generator.png)

## Globale Parameter und Imports

In [1]:

import json
import socket
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "widget")
except Exception:
    pass

FPC_IP = "192.168.1.10"
FPC_PORT = 5555
SOCKET_TIMEOUT = 5.0
SCREENSHOT_TIMEOUT = 10.0
TRACE_READ_MAX_BYTES = 256 * 1024

REPLAY = True
REPLAY_FILE = Path("recordings") / "6-fpc1500_sa_mixer_replay.json"
SCREENSHOT_DIR = Path("screenshots")
SCREENSHOT_FILENAME = "screen.png"

## SCPI und Plot Hilfsfunktionen, File Save /Replay Funktionen

In [2]:



def scpi_query(host: str, port: int, cmd: str, max_bytes: int = 4096) -> str:
    cmd = cmd.strip() + "\n"
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(SOCKET_TIMEOUT)
    try:
        s.connect((host, port))
        s.sendall(cmd.encode())
        buf = b""
        while len(buf) < max_bytes:
            chunk = s.recv(8192)
            if not chunk:
                break
            buf += chunk
            if b"\n" in buf:
                break
        return buf.decode("utf-8", errors="replace").strip()
    finally:
        s.close()


def get_trace_data(host: str, port: int):
    """Frequenzachse (Hz), Trace-Werte (dBm) und RBW (Hz)."""
    try:
        trace_s = scpi_query(host, port, "TRAC:DATA? TRACE1", max_bytes=TRACE_READ_MAX_BYTES)
    except Exception:
        return None
    amps = []
    for part in trace_s.replace(",", " ").split():
        try:
            amps.append(float(part))
        except ValueError:
            continue
    if not amps:
        return None
    try:
        star_s = scpi_query(host, port, "FREQ:STAR?", max_bytes=256)
        stop_s = scpi_query(host, port, "FREQ:STOP?", max_bytes=256)
        freq_start = float(star_s.strip())
        freq_stop = float(stop_s.strip())
    except Exception:
        try:
            cent_s = scpi_query(host, port, "FREQ:CENT?", max_bytes=256)
            span_s = scpi_query(host, port, "FREQ:SPAN?", max_bytes=256)
            cent = float(cent_s.strip())
            span = float(span_s.strip())
            freq_start = cent - span / 2
            freq_stop = cent + span / 2
        except Exception:
            return None
    rbw_hz = None
    for rbw_cmd in ["BAND:RES?", "BANDwidth:RESolution?"]:
        try:
            rbw_hz = float(scpi_query(host, port, rbw_cmd, max_bytes=256).strip())
            break
        except Exception:
            continue
    n = len(amps)
    freqs = [freq_start + (freq_stop - freq_start) * i / max(1, n - 1) for i in range(n)]
    return (freqs, amps, rbw_hz)


def screenshot_save(host: str, port: int, filename: str = "screen.png") -> str | None:
    commands = ["HCOP:DEV:LANG PNG", "HCOP:DEST 'MMEM'", f"MMEM:NAME '{filename}'", "HCOP:IMM"]
    cmd = "\n".join(commands) + "\n"
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(SCREENSHOT_TIMEOUT)
    try:
        s.connect((host, port))
        s.sendall(cmd.encode())
        buf = b""
        while len(buf) < 4096:
            try:
                chunk = s.recv(1024)
                if not chunk:
                    break
                buf += chunk
                if b"\n" in buf:
                    break
            except socket.timeout:
                break
        reply = buf.decode("utf-8", errors="replace").strip()
        if reply and "error" in reply.lower():
            return reply
        return None
    except Exception as e:
        return str(e)
    finally:
        s.close()


def save_replay_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def load_replay_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Replay-Datei nicht gefunden: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def screenshot_read(host: str, port: int, filename: str) -> bytes | None:
    cmd = f"MMEM:DATA? '{filename}'\n"
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(SCREENSHOT_TIMEOUT)
    try:
        s.connect((host, port))
        s.sendall(cmd.encode())
        buf = b""
        while b"#" not in buf and len(buf) < 1024:
            chunk = s.recv(256)
            if not chunk:
                return None
            buf += chunk
        if b"#" not in buf:
            return None
        start = buf.index(b"#")
        buf = buf[start:]
        if len(buf) < 2:
            buf += s.recv(2 - len(buf))
        n_digits = int(chr(buf[1]))
        if n_digits < 1 or n_digits > 9:
            return None
        while len(buf) < 2 + n_digits:
            buf += s.recv(2 + n_digits - len(buf))
        data_len = int(buf[2 : 2 + n_digits].decode())
        buf = buf[2 + n_digits :]
        while len(buf) < data_len:
            chunk = s.recv(min(65536, data_len - len(buf)))
            if not chunk:
                break
            buf += chunk
        return buf[:data_len] if len(buf) >= data_len else None
    except Exception:
        return None
    finally:
        s.close()


def plot_mixer_spectrum(
    freq_hz: np.ndarray,
    p_dbm: np.ndarray,
    rbw_hz,
    title: str,
    fig_id: str,
):
    """Spektrum mit Marker. Matplotlib Slider + ipympl fuehren in Jupyter oft zu eingefrorenen Widgets;
    daher ipywidgets.IntSlider (Trace-Bin), continuous_update=False (Update beim Loslassen)."""
    _ipy_interactive = plt.isinteractive()
    plt.ioff()
    plt.close(fig_id)
    freq_mhz = np.asarray(freq_hz, dtype=float) / 1e6
    p_dbm = np.asarray(p_dbm, dtype=float)
    rbw_txt = f" | RBW={rbw_hz:.1f} Hz" if rbw_hz is not None else " | RBW=n/a"
    mode = "Replay" if REPLAY else "Live"
    n = int(p_dbm.size)

    fig = plt.figure(figsize=(12.0, 5.8), num=fig_id)
    ax = fig.add_subplot(1, 1, 1)

    ax.plot(freq_mhz, p_dbm, color="C0", lw=1.0)
    ax.set_xlabel("Frequenz (MHz)")
    ax.set_ylabel("Leistung (dBm)")
    ax.set_title(f"{title} ({mode})" + rbw_txt)
    ax.grid(True, alpha=0.35)

    y0, y1 = float(np.min(p_dbm)), float(np.max(p_dbm))
    if np.isfinite(y0) and np.isfinite(y1) and (y1 > y0):
        pad = max(1.0, 0.05 * (y1 - y0))
        ax.set_ylim(y0 - pad, y1 + pad)

    info_text = ax.text(
        0.01,
        0.02,
        "",
        transform=ax.transAxes,
        fontsize=10,
        ha="left",
        va="bottom",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85),
    )

    if n >= 2 and widgets is not None:
        idx0 = int(np.argmin(np.abs(freq_mhz - np.mean(freq_mhz))))
        m_peak, = ax.plot([freq_mhz[idx0]], [p_dbm[idx0]], "o", color="red", ms=7)

        def set_idx(i):
            i = max(0, min(n - 1, int(i)))
            m_peak.set_data([freq_mhz[i]], [p_dbm[i]])
            info_text.set_text(
                f"f={freq_mhz[i]:.6f} MHz | P={p_dbm[i]:.2f} dBm | Index={i}"
            )

        def on_idx(change):
            set_idx(change["new"])

        idx_slider = widgets.IntSlider(
            value=idx0,
            min=0,
            max=n - 1,
            step=1,
            description="Trace-Bin",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="95%"),
        )

        btn_minus = widgets.Button(
            description="-1",
            tooltip="Marker um eine Position nach links",
            layout=widgets.Layout(width="70px"),
        )
        btn_plus = widgets.Button(
            description="+1",
            tooltip="Marker um eine Position nach rechts",
            layout=widgets.Layout(width="70px"),
        )

        def on_minus(_):
            idx_slider.value = max(idx_slider.min, idx_slider.value - 1)

        def on_plus(_):
            idx_slider.value = min(idx_slider.max, idx_slider.value + 1)

        btn_minus.on_click(on_minus)
        btn_plus.on_click(on_plus)
        idx_slider.observe(on_idx, names="value")

        set_idx(idx0)
        nav_row = widgets.HBox([btn_minus, btn_plus])
        display(widgets.VBox([fig.canvas, idx_slider, nav_row]))
    elif n >= 2:
        idx0 = int(np.argmin(np.abs(freq_mhz - np.mean(freq_mhz))))
        ax.plot([freq_mhz[idx0]], [p_dbm[idx0]], "o", color="red", ms=7)
        info_text.set_text(
            f"f={freq_mhz[idx0]:.6f} MHz | P={p_dbm[idx0]:.2f} dBm | Index={idx0} — pip install ipywidgets"
        )
        fig.canvas.draw_idle()
        plt.show()
    else:
        info_text.set_text(
            f"f={freq_mhz[0]:.6f} MHz | P={p_dbm[0]:.2f} dBm" if n == 1 else "Keine Daten"
        )
        fig.canvas.draw_idle()
        plt.show()

    if n > 0:
        print(
            f"Punkte: {n}, f = {freq_mhz[0]:.4f} … {freq_mhz[-1]:.4f} MHz"
        )
    if _ipy_interactive:
        plt.ion()



def mixer_merge_save(key: str, freqs: list, amps: list, rbw_hz, idn: str):
    if REPLAY_FILE.exists():
        data = load_replay_json(REPLAY_FILE)
    else:
        data = {"meta": {"type": "fpc1500_sa_mixer", "version": 1}}
    data["idn"] = idn
    data[key] = {"freqs_hz": list(freqs), "amps": list(amps), "rbw_hz": rbw_hz}
    save_replay_json(REPLAY_FILE, data)
    verify = load_replay_json(REPLAY_FILE)
    counts = {}
    for k in ("spectrum_lo", "spectrum_if", "spectrum_rf"):
        blk = verify.get(k)
        counts[k] = len(blk.get("amps", [])) if isinstance(blk, dict) else 0
    print("Replay gespeichert:", REPLAY_FILE.resolve())
    print("  Spektren im File (Anzahl Samples):", counts)


print("Konfiguration:", FPC_IP, "Port", FPC_PORT)
print(f"REPLAY={REPLAY}, Datei={REPLAY_FILE}")


Konfiguration: 192.168.1.10 Port 5555
REPLAY=True, Datei=recordings\6-fpc1500_sa_mixer_replay.json


### Verbindung testen (`*IDN?`)

Prüfen, ob der FPC1500 per SCPI erreichbar ist.


In [3]:
if REPLAY:
    try:
        replay = load_replay_json(REPLAY_FILE)
        idn_reply = replay.get("idn", "REPLAY")
        print("Gerät (Replay):", idn_reply)
    except FileNotFoundError:
        idn_reply = "REPLAY (keine Datei)"
        print("Replay-Datei noch nicht vorhanden — bitte später Live-Messungen ausführen.")
else:
    idn_reply = scpi_query(FPC_IP, FPC_PORT, "*IDN?")
    print("Gerät:", idn_reply if idn_reply else "Keine Antwort (IP/Port prüfen?)")


Gerät (Replay): Rohde&Schwarz,FPC1500,1328.6660K03/102548,V1.50


### Messung 1 — Spektrum des **LO** (Lokaloszillator)

**Messablauf**

1. Aufbau so einstellen, dass der **LO-Pfad** zum Spektrumanalysator geführt wird (wie im Praktikum vorgegeben).
2. Am FPC1500 **Spectrum Analyzer** wählen; **Span / Center** so setzen, dass der **LO-Peak** im Fenster liegt.
3. Warten bis der Trace stabil ist, dann **diese Code-Zelle** ausführen.

Die Daten werden unter dem Schlüssel `spectrum_lo` in der Replay-Datei abgelegt.


In [4]:
key = "spectrum_lo"
title = "Messung 1: LO-Spektrum"

if REPLAY:
    data = load_replay_json(REPLAY_FILE)
    blk = data.get(key)
    if not blk:
        raise RuntimeError("Replay ohne spectrum_lo — Messung 1 fehlt oder alte Datei.")
    freqs = blk["freqs_hz"]
    amps = blk["amps"]
    rbw_hz = blk.get("rbw_hz")
    print("Replay:", key, "—", len(amps), "Samples")
else:
    result = get_trace_data(FPC_IP, FPC_PORT)
    if result is None:
        raise RuntimeError("Keine Trace-Daten — Mode SA? Span? Trigger?")
    freqs, amps, rbw_hz = result
    mixer_merge_save(key, freqs, amps, rbw_hz, idn_reply)

freq_hz = np.asarray(freqs, dtype=float)
p_dbm = np.asarray(amps, dtype=float)
plot_mixer_spectrum(freq_hz, p_dbm, rbw_hz, title, fig_id="mixer_lo")


Replay: spectrum_lo — 1183 Samples


Punkte: 1183, f = 424.8376 … 442.8376 MHz


### Messung 2 — Spektrum des **IF-Signals**

**Messablauf**

1. Messpunkt/Messkoppler auf das **IF-Signal** umlegen (gleicher Messmodus am FPC: Spektrum).
2. Span/Center an die **IF-Frequenz** anpassen; Trace stabil.
3. **Diese Zelle ausführen** (speichert `spectrum_if`).

> Die drei Messungen bauen aufeinander auf — bitte **ohne lange Pause** nacheinander durchführen,
> damit Geräteeinstellungen und Beschriftung zusammenpassen.


In [5]:
key = "spectrum_if"
title = "Messung 2: IF-Spektrum"

if REPLAY:
    data = load_replay_json(REPLAY_FILE)
    blk = data.get(key)
    if not blk:
        raise RuntimeError("Replay ohne spectrum_if.")
    freqs = blk["freqs_hz"]
    amps = blk["amps"]
    rbw_hz = blk.get("rbw_hz")
    print("Replay:", key, "—", len(amps), "Samples")
else:
    result = get_trace_data(FPC_IP, FPC_PORT)
    if result is None:
        raise RuntimeError("Keine Trace-Daten.")
    freqs, amps, rbw_hz = result
    mixer_merge_save(key, freqs, amps, rbw_hz, idn_reply)

freq_hz = np.asarray(freqs, dtype=float)
p_dbm = np.asarray(amps, dtype=float)
plot_mixer_spectrum(freq_hz, p_dbm, rbw_hz, title, fig_id="mixer_if")


Replay: spectrum_if — 1183 Samples


Punkte: 1183, f = 2.5000 … 7.5000 MHz


### Messung 3 — Spektrum am **RF-Ausgang** des Mischers

**Messablauf**

1. Messung am **RF-Ausgang** (bzw. nach dem Mischer in der RF-Kette) — Span so wählen, dass **LO-Leck**, **IF-Mischprodukt**
   und ggf. **RF-Träger** im Spektrum beobachtet werden können (exakte Lage hängt vom Aufbau ab).
2. Trace stabil, dann **diese Zelle ausführen** (speichert `spectrum_rf`).

Damit liegen alle drei Spektren in einer gemeinsamen Replay-Datei für die Auswertung.


In [6]:
key = "spectrum_rf"
title = "Messung 3: RF-Ausgang (Spektrum)"

if REPLAY:
    data = load_replay_json(REPLAY_FILE)
    blk = data.get(key)
    if not blk:
        raise RuntimeError("Replay ohne spectrum_rf.")
    freqs = blk["freqs_hz"]
    amps = blk["amps"]
    rbw_hz = blk.get("rbw_hz")
    print("Replay:", key, "—", len(amps), "Samples")
else:
    result = get_trace_data(FPC_IP, FPC_PORT)
    if result is None:
        raise RuntimeError("Keine Trace-Daten.")
    freqs, amps, rbw_hz = result
    mixer_merge_save(key, freqs, amps, rbw_hz, idn_reply)

freq_hz = np.asarray(freqs, dtype=float)
p_dbm = np.asarray(amps, dtype=float)
plot_mixer_spectrum(freq_hz, p_dbm, rbw_hz, title, fig_id="mixer_rf")


Replay: spectrum_rf — 1183 Samples


Punkte: 1183, f = 383.7614 … 483.7614 MHz


## Aufgaben zur Auswertung (nach allen drei Messungen)

Nutze die gespeicherten Spektren (`spectrum_lo`, `spectrum_if`, `spectrum_rf` in der Replay-JSON) bzw. die Plots oben.

1. **Pegel von LO und IF**  
   Bestimme die **Pegel in dBm** für LO und IF (z. B. mit Marker/Peak in den gespeicherten Traces oder über den Maximalwert im relevanten Frequenzbereich).

2. **Dämpfung des LO am RF-Ausgang**  
   Vergleiche den LO-Pegel (Messung 1) mit dem LO-Leck bzw. LO-Anteil im Spektrum von Messung 3 am RF-Ausgang — ermittle die **Dämpfung** (Differenz in dB).

3. **Conversion Loss**  
   Bestimme, wie stark das **IF-Signal als Mischprodukt am RF-Ausgang** erscheint: Vergleich IF-Pegel (Messung 2) mit dem entsprechenden IF-Mischprodukt im Spektrum von Messung 3 (gleiche Frequenz). Der **Conversion Loss** wird üblicherweise als Differenz der dBm-Pegel angegeben (ggf. unter Berücksichtigung von Referenzimpedanz/Kalibrierung laut Praktikumsanleitung).

**Hinweis:** Es sind **drei Messungen in Abfolge** durchzuführen; erst danach sind alle Daten für die Punkte 1–3 vollständig.


In [22]:
# 1. Pegel bestimmen
# LO
data_lo = load_replay_json(REPLAY_FILE)
blk_lo = data_lo.get("spectrum_lo")
amps_lo = blk_lo["amps"]
level_lo_dbm = np.max(amps_lo)

print(f"Pegel LO = {level_lo_dbm:.2f} dBm")

# IF
data_if = load_replay_json(REPLAY_FILE)
blk_if = data_if.get("spectrum_if")
amps_if = blk_if["amps"]
freqs_if = np.array(blk_if["freqs_hz"])
level_if_dbm = np.max(amps_if)
idx_if_in = np.argmax(amps_if)
f_if = freqs_if[idx_if_in]

print(f"Pegel IF = {level_if_dbm:.2f} dBm")


# 2. Dämpfung LO Pegel
level_lo_rf_dbm = -36.96 # aus Plot abgelesen
damp_lo_db = level_lo_rf_dbm - level_lo_dbm

print(f"Dämpfung LO = {damp_lo_db:.2f} dB")


# 3. Conversion Loss
data_rf = load_replay_json(REPLAY_FILE)
blk_rf = data_rf.get("spectrum_rf") 
amps_rf = np.array(blk_rf["amps"])
freqs_rf = np.array(blk_rf["freqs_hz"])

level_if_rf_dbm = -20.94 # aus Plot abgelesen

conv_loss_db = level_if_dbm - level_if_rf_dbm

print(f"Conversion Loss = {conv_loss_db:.2f} dB")

Pegel LO = 2.15 dBm
Pegel IF = -16.24 dBm
Dämpfung LO = -39.11 dB
Conversion Loss = 4.70 dB


### Screenshot vom Gerät

Hardcopy (PNG) auf dem FPC auslösen, dann Datei per `MMEM:DATA?` lesen und anzeigen (wie in den anderen SA-Notebooks).
Kann einige Sekunden dauern.


In [ ]:
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

if REPLAY:
    replay = load_replay_json(REPLAY_FILE)
    shot_path_str = replay.get("screenshot_path", "")
    shot_path = Path(shot_path_str) if shot_path_str else None
    if shot_path and shot_path.exists():
        print("Replay-Screenshot geladen:", shot_path)
        display(Image(filename=str(shot_path)))
    else:
        print("REPLAY=True: Kein gültiger screenshot_path in Replay-Datei.")
        if shot_path_str:
            print("Gespeicherter Pfad (nicht gefunden):", shot_path_str)
else:
    err = screenshot_save(FPC_IP, FPC_PORT, SCREENSHOT_FILENAME)
    if err:
        print("Screenshot (HCOP) Fehler:", err)
    else:
        png_bytes = screenshot_read(FPC_IP, FPC_PORT, SCREENSHOT_FILENAME)
        if png_bytes:
            path = SCREENSHOT_DIR / f"{REPLAY_FILE.stem.replace('_replay', '')}_screenshot.png"
            path.write_bytes(png_bytes)
            print("Gespeichert:", path.resolve())
            display(Image(data=png_bytes))
            try:
                replay = load_replay_json(REPLAY_FILE) if REPLAY_FILE.exists() else {}
                replay["screenshot_path"] = str(path)
                save_replay_json(REPLAY_FILE, replay)
                print("Replay aktualisiert mit screenshot_path:", REPLAY_FILE)
            except Exception as ex:
                print("screenshot_path konnte nicht gespeichert werden:", ex)
        else:
            print("Screenshot auf Gerät gespeichert; Lesen nicht möglich.")
